In [11]:
import os
import io

import numpy as np

from typing import Tuple

import time
import cv2

from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torchvision
import torch.onnx
import torchsummary
print('torch.__version__', torch.__version__)
print('torchvision.__version__', torchvision.__version__)

import onnx
import onnx2keras
import onnxruntime
from onnxsim import simplify
from onnx_tf.backend import prepare
print('onnx.__version__', onnx.__version__)

# import tvm
# import tvm.relay
# import tvm.contrib.graph_runtime as graph_runtime

from mobilenet_v2_tsm_stateful import MobileNetV2  # !!!

import tensorflow as tf
print('tf.__version__', tf.__version__)
from tensorflow.python.keras import layers
from tensorflow.python.keras.engine import training
from keras.models import load_model
print('tf.keras.__version__', tf.keras.__version__)

import tensorflowjs as tfjs
print('tfjs.__version__', tfjs.__version__)

# import warnings
# warnings.filterwarnings('ignore')

# os.makedirs('./stateful_models/')
MODELS_ROOT = './stateful_models'

torch.__version__ 1.4.0
torchvision.__version__ 0.5.0


/Users/izakharkin/Desktop/skoltech/vrarhaptics/deepjest/phynder/convert/onnx-tensorflow/onnx_tf/common/__init__.py:96: UserWarning: onnx_tf.common.get_outputs_names is deprecated. It will be removed in future release. Use TensorflowGraph.get_outputs_names instead.
  warnings.warn(message)
Using TensorFlow backend.


onnx.__version__ 1.6.0
tf.__version__ 2.2.0
tf.keras.__version__ 2.3.0-tf
tfjs.__version__ 2.0.1


* torch2onnx:

In [12]:
SOFTMAX_THRES = 0
HISTORY_LOGIT = True
REFINE_OUTPUT = True

# def torch2tvm_module(torch_module: torch.nn.Module, torch_inputs: Tuple[torch.Tensor, ...], target):
#     torch_module.eval()
#     input_names = []
#     input_shapes = {}
#     with torch.no_grad():
#         for index, torch_input in enumerate(torch_inputs):
#             name = "i" + str(index)
#             input_names.append(name)
#             input_shapes[name] = torch_input.shape
#         buffer = io.BytesIO()
#         torch.onnx.export(torch_module, torch_inputs, buffer, input_names=input_names, output_names=["o" + str(i) for i in range(len(torch_inputs))])
#         outs = torch_module(*torch_inputs)
#         buffer.seek(0, 0)
#         onnx_model = onnx.load_model(buffer)
#         relay_module, params = tvm.relay.frontend.from_onnx(onnx_model, shape=input_shapes)
#     with tvm.relay.build_config(opt_level=3):
#         graph, tvm_module, params = tvm.relay.build(relay_module, target, params=params)
#     return graph, tvm_module, params


# def torch2executor(torch_module: torch.nn.Module, torch_inputs: Tuple[torch.Tensor, ...], target):
#     prefix = f"mobilenet_tsm_tvm_{target}"
#     lib_fname = f'{prefix}.tar'
#     graph_fname = f'{prefix}.json'
#     params_fname = f'{prefix}.params'
#     if os.path.exists(lib_fname) and os.path.exists(graph_fname) and os.path.exists(params_fname):
#         with open(graph_fname, 'rt') as f:
#             graph = f.read()
#         tvm_module = tvm.module.load(lib_fname)
#         params = tvm.relay.load_param_dict(bytearray(open(params_fname, 'rb').read()))
#     else:
#         graph, tvm_module, params = torch2tvm_module(torch_module, torch_inputs, target)
#         tvm_module.export_library(lib_fname)
#         with open(graph_fname, 'wt') as f:
#             f.write(graph)
#         with open(params_fname, 'wb') as f:
#             f.write(tvm.relay.save_param_dict(params))

#     ctx = tvm.gpu() if target.startswith('cuda') else tvm.cpu()
#     graph_module = graph_runtime.create(graph, tvm_module, ctx)
#     for pname, pvalue in params.items():
#         graph_module.set_input(pname, pvalue)

#     def executor(inputs: Tuple[tvm.nd.NDArray]):
#         for index, value in enumerate(inputs):
#             graph_module.set_input(index, value)
#         graph_module.run()
#         return tuple(graph_module.get_output(index) for index in range(len(inputs)))

#     return executor, ctx


# def get_executor(use_gpu=True):
#     torch_module = MobileNetV2(n_class=27)
#     if not os.path.exists("mobilenetv2_jester_online.pth.tar"):  # checkpoint not downloaded
#         print('Downloading PyTorch checkpoint...')
#         import urllib.request
#         url = 'https://file.lzhu.me/projects/tsm/models/mobilenetv2_jester_online.pth.tar'
#         urllib.request.urlretrieve(url, './mobilenetv2_jester_online.pth.tar')
#     torch_module.load_state_dict(torch.load("mobilenetv2_jester_online.pth.tar"))
#     torch_inputs = (torch.rand(1, 3, 224, 224),
#                     torch.zeros([1, 3, 56, 56]),
#                     torch.zeros([1, 4, 28, 28]),
#                     torch.zeros([1, 4, 28, 28]),
#                     torch.zeros([1, 8, 14, 14]),
#                     torch.zeros([1, 8, 14, 14]),
#                     torch.zeros([1, 8, 14, 14]),
#                     torch.zeros([1, 12, 14, 14]),
#                     torch.zeros([1, 12, 14, 14]),
#                     torch.zeros([1, 20, 7, 7]),
#                     torch.zeros([1, 20, 7, 7]))
#     if use_gpu:
#         target = 'cuda'
#     else:
#         target = 'llvm -mcpu=cortex-a72 -target=armv7l-linux-gnueabihf'
#     return torch2executor(torch_module, torch_inputs, target)


def transform(frame: np.ndarray):
    # 480, 640, 3, 0 ~ 255
    frame = cv2.resize(frame, (224, 224))  # (224, 224, 3) 0 ~ 255
    frame = frame / 255.0  # (224, 224, 3) 0 ~ 1.0
    frame = np.transpose(frame, axes=[2, 0, 1])  # (3, 224, 224) 0 ~ 1.0
    frame = np.expand_dims(frame, axis=0)  # (1, 3, 480, 640) 0 ~ 1.0
    return frame


class GroupScale(object):
    """ Rescales the input PIL.Image to the given 'size'.
    'size' will be the size of the smaller edge.
    For example, if height > width, then image will be
    rescaled to (size * height / width, size)
    size: size of the smaller edge
    interpolation: Default: PIL.Image.BILINEAR
    """

    def __init__(self, size, interpolation=Image.BILINEAR):
        self.worker = torchvision.transforms.Scale(size, interpolation)

    def __call__(self, img_group):
        return [self.worker(img) for img in img_group]


class GroupCenterCrop(object):
    def __init__(self, size):
        self.worker = torchvision.transforms.CenterCrop(size)

    def __call__(self, img_group):
        return [self.worker(img) for img in img_group]


class Stack(object):

    def __init__(self, roll=False):
        self.roll = roll

    def __call__(self, img_group):
        if img_group[0].mode == 'L':
            return np.concatenate([np.expand_dims(x, 2) for x in img_group], axis=2)
        elif img_group[0].mode == 'RGB':
            if self.roll:
                return np.concatenate([np.array(x)[:, :, ::-1] for x in img_group], axis=2)
            else:
                return np.concatenate(img_group, axis=2)


class ToTorchFormatTensor(object):
    """ Converts a PIL.Image (RGB) or numpy.ndarray (H x W x C) in the range [0, 255]
    to a torch.FloatTensor of shape (C x H x W) in the range [0.0, 1.0] """

    def __init__(self, div=True):
        self.div = div

    def __call__(self, pic):
        if isinstance(pic, np.ndarray):
            # handle numpy array
            img = torch.from_numpy(pic).permute(2, 0, 1).contiguous()
        else:
            # handle PIL Image
            img = torch.ByteTensor(torch.ByteStorage.from_buffer(pic.tobytes()))
            img = img.view(pic.size[1], pic.size[0], len(pic.mode))
            # put it from HWC to CHW format
            # yikes, this transpose takes 80% of the loading time/CPU
            img = img.transpose(0, 1).transpose(0, 2).contiguous()
        return img.float().div(255) if self.div else img.float()


class GroupNormalize(object):
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        rep_mean = self.mean * (tensor.size()[0] // len(self.mean))
        rep_std = self.std * (tensor.size()[0] // len(self.std))

        # TODO: make efficient
        for t, m, s in zip(tensor, rep_mean, rep_std):
            t.sub_(m).div_(s)

        return tensor


def get_transform():
    cropping = torchvision.transforms.Compose([
        GroupScale(256),
        GroupCenterCrop(224),
    ])
    transform = torchvision.transforms.Compose([
        cropping,
        Stack(roll=False),
        ToTorchFormatTensor(div=True),
        GroupNormalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return transform

categories = [
    "Doing other things",  # 0
    "Drumming Fingers",  # 1
    "No gesture",  # 2
    "Pulling Hand In",  # 3
    "Pulling Two Fingers In",  # 4
    "Pushing Hand Away",  # 5
    "Pushing Two Fingers Away",  # 6
    "Rolling Hand Backward",  # 7
    "Rolling Hand Forward",  # 8
    "Shaking Hand",  # 9
    "Sliding Two Fingers Down",  # 10
    "Sliding Two Fingers Left",  # 11
    "Sliding Two Fingers Right",  # 12
    "Sliding Two Fingers Up",  # 13
    "Stop Sign",  # 14
    "Swiping Down",  # 15
    "Swiping Left",  # 16
    "Swiping Right",  # 17
    "Swiping Up",  # 18
    "Thumb Down",  # 19
    "Thumb Up",  # 20
    "Turning Hand Clockwise",  # 21
    "Turning Hand Counterclockwise",  # 22
    "Zooming In With Full Hand",  # 23
    "Zooming In With Two Fingers",  # 24
    "Zooming Out With Full Hand",  # 25
    "Zooming Out With Two Fingers"  # 26
]


n_still_frame = 0

def process_output(idx_, history):
    # idx_: the output of current frame
    # history: a list containing the history of predictions
    if not REFINE_OUTPUT:
        return idx_, history

    max_hist_len = 20  # max history buffer

    # mask out illegal action
    if idx_ in [7, 8, 21, 22, 3]:
        idx_ = history[-1]

    # use only single no action class
    if idx_ == 0:
        idx_ = 2
    
    # history smoothing
    if idx_ != history[-1]:
        if not (history[-1] == history[-2]): #  and history[-2] == history[-3]):
            idx_ = history[-1]
    

    history.append(idx_)
    history = history[-max_hist_len:]

    return history[-1], history

In [13]:
def torch2onnx(
    torch_module: torch.nn.Module, 
    torch_inputs: Tuple[torch.Tensor, ...], 
    onnx_path
):
    torch_module.eval()
    input_names = []
    input_shapes = {}
    with torch.no_grad():
        for index, torch_input in enumerate(torch_inputs):
            name = "i" + str(index)
            input_names.append(name)
            input_shapes[name] = torch_input.shape
        with open(onnx_path, 'wb') as model_file:
            torch.onnx.export(
                torch_module, 
                torch_inputs, 
                model_file, 
                input_names=input_names, 
                output_names=["o" + str(i) for i in range(len(torch_inputs))],
                opset_version=10
            )

* Load the model:

In [14]:
os.makedirs('./models', exist_ok=True)
TORCH_MODEL_PATH= './models/mobilenetv2_jester_online.pth.tar'
torch_module = MobileNetV2(n_class=27)
if not os.path.exists(TORCH_MODEL_PATH):  # checkpoint not downloaded
    print('Downloading PyTorch checkpoint...')
    import urllib.request
    url = 'https://file.lzhu.me/projects/tsm/models/mobilenetv2_jester_online.pth.tar'
    urllib.request.urlretrieve(url, TORCH_MODEL_PATH)
torch_module.load_state_dict(torch.load(TORCH_MODEL_PATH))
torch_module.eval()

MobileNetV2(
  (features): ModuleList(
    (0): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
        (3): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
       

In [15]:
TORCH_STATEFUL_MODEL_PATH = f'{MODELS_ROOT}/jestnet_stateful_torch.pth'
with open(TORCH_STATEFUL_MODEL_PATH, 'wb') as file:
    torch.save(torch_module, file)

In [16]:
test_module = torch.load(TORCH_STATEFUL_MODEL_PATH, map_location=torch.device('cpu'))
test_module

MobileNetV2(
  (features): ModuleList(
    (0): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
        (3): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
       

* Convert to ONNX:

In [17]:
ONNX_MODEL_PATH = f'{MODELS_ROOT}/jestnet_stateful.onnx'

torch_inputs = (torch.rand(1, 3, 224, 224, dtype=torch.float32))

for param in torch_module.parameters():
    param = param.float()

for module in torch_module.children():
    for module_1 in module.children():
        for module_2 in module_1.children():
            for module_3 in module_2.children():
                if hasattr(module_3, 'num_batches_tracked'):
                    module_3.num_batches_tracked = module_3.num_batches_tracked.float()
#                 if str(module_3).split('(')[0] == 'BatchNorm2d':
#                     print(module_3)

torch2onnx(
    torch_module=torch_module, 
    torch_inputs=torch_inputs, 
    onnx_path=ONNX_MODEL_PATH
)

* Load from ONNX:

In [18]:
with open(ONNX_MODEL_PATH, 'rb') as onnx_model_file:
    onnx_model = onnx.load_model(onnx_model_file)
onnx.checker.check_model(onnx_model)

In [19]:
print(onnx.helper.printable_graph(onnx_model.graph))

graph torch-jit-export (
  %i0[FLOAT, 1x3x224x224]
) initializers (
  %classifier.bias[FLOAT, 27]
  %classifier.weight[FLOAT, 27x1280]
  %features.0.0.weight[FLOAT, 32x3x3x3]
  %features.0.1.bias[FLOAT, 32]
  %features.0.1.num_batches_tracked[INT64, scalar]
  %features.0.1.running_mean[FLOAT, 32]
  %features.0.1.running_var[FLOAT, 32]
  %features.0.1.weight[FLOAT, 32]
  %features.1.conv.0.weight[FLOAT, 32x1x3x3]
  %features.1.conv.1.bias[FLOAT, 32]
  %features.1.conv.1.num_batches_tracked[FLOAT, scalar]
  %features.1.conv.1.running_mean[FLOAT, 32]
  %features.1.conv.1.running_var[FLOAT, 32]
  %features.1.conv.1.weight[FLOAT, 32]
  %features.1.conv.3.weight[FLOAT, 16x32x1x1]
  %features.1.conv.4.bias[FLOAT, 16]
  %features.1.conv.4.num_batches_tracked[FLOAT, scalar]
  %features.1.conv.4.running_mean[FLOAT, 16]
  %features.1.conv.4.running_var[FLOAT, 16]
  %features.1.conv.4.weight[FLOAT, 16]
  %features.10.conv.0.weight[FLOAT, 384x64x1x1]
  %features.10.conv.1.bias[FLOAT, 384]
  %featur

* [optional] Simplify:

In [20]:
model_simp, check = simplify(onnx_model)
assert check, "Simplified ONNX model could not be validated"

In [21]:
print('Before', onnx_model.ByteSize())
print('After', model_simp.ByteSize())

Before 9315993
After 9087822


In [22]:
print(onnx.helper.printable_graph(model_simp.graph))

graph torch-jit-export (
  %i0[FLOAT, 1x3x224x224]
) initializers (
  %classifier.bias[FLOAT, 27]
  %classifier.weight[FLOAT, 27x1280]
  %341[INT64, 1]
  %343[FLOAT, 1x3x56x56]
  %372[INT64, 1]
  %374[FLOAT, 1x4x28x28]
  %395[INT64, 1]
  %397[FLOAT, 1x4x28x28]
  %426[INT64, 1]
  %428[FLOAT, 1x8x14x14]
  %449[INT64, 1]
  %451[FLOAT, 1x8x14x14]
  %472[INT64, 1]
  %474[FLOAT, 1x8x14x14]
  %503[INT64, 1]
  %505[FLOAT, 1x12x14x14]
  %526[INT64, 1]
  %528[FLOAT, 1x12x14x14]
  %557[INT64, 1]
  %559[FLOAT, 1x20x7x7]
  %580[INT64, 1]
  %582[FLOAT, 1x20x7x7]
  %668[FLOAT, 32x3x3x3]
  %670[FLOAT, 32]
  %672[FLOAT, 32x1x3x3]
  %674[FLOAT, 32]
  %676[FLOAT, 16x32x1x1]
  %678[FLOAT, 16]
  %680[FLOAT, 96x16x1x1]
  %682[FLOAT, 96]
  %684[FLOAT, 96x1x3x3]
  %686[FLOAT, 96]
  %688[FLOAT, 24x96x1x1]
  %690[FLOAT, 24]
  %692[FLOAT, 144x24x1x1]
  %694[FLOAT, 144]
  %696[FLOAT, 144x1x3x3]
  %698[FLOAT, 144]
  %700[FLOAT, 24x144x1x1]
  %702[FLOAT, 24]
  %704[FLOAT, 144x24x1x1]
  %706[FLOAT, 144]
  %708[FLOAT

In [23]:
ONNX_SIMPLE_MODEL_PATH = f'{MODELS_ROOT}/jestnet_stateful_simple.onnx'
onnx.save(model_simp, ONNX_SIMPLE_MODEL_PATH)

* ONNX runtime check:

In [24]:
ort_session = onnxruntime.InferenceSession(ONNX_SIMPLE_MODEL_PATH)

In [25]:
input_names = [ort_session.get_inputs()[i].name for i in range(len(ort_session.get_inputs()))]
input_names

['i0']

In [26]:
output_names = [ort_session.get_outputs()[i].name for i in range(len(ort_session.get_inputs()))]
output_names

['o0']

In [27]:
np_inputs = [
    np.random.rand(1, 3, 224, 224)
]
np_inputs = [np_input.astype(np.float32) for np_input in np_inputs]

In [28]:
%%time
outputs = ort_session.run(output_names, {input_names[i]: np_inputs[i] for i in range(len(np_inputs))})

CPU times: user 21 ms, sys: 15.9 ms, total: 36.8 ms
Wall time: 9.1 ms


In [29]:
for i in range(len(output_names)):
    print(outputs[i].shape)

(1, 27)


* TFLite -> ONNX (for ONNX runtime test):

In [30]:
import tflite2onnx

tflite_path = '/Users/izakharkin/Desktop/inclusio/Inclusio/mediapipe/mediapipe/models/hand_landmark.tflite'
onnx_path = './models/hand_landmark.onnx'

tflite2onnx.convert(tflite_path, onnx_path)

NotImplementedError: Unsupported TFLite OP: 6

* ONNX -> TensorFlow:

In [31]:
tf_rep = prepare(onnx_model)

2020-07-05 21:54:26,009 - onnx-tf - INFO - Fail to get since_version of BitShift in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:54:26,009 - onnx-tf - INFO - Unknown op ConstantFill in domain `ai.onnx`.
2020-07-05 21:54:26,010 - onnx-tf - INFO - Fail to get since_version of CumSum in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:54:26,011 - onnx-tf - INFO - Fail to get since_version of Det in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:54:26,012 - onnx-tf - INFO - Fail to get since_version of DynamicQuantizeLinear in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:54:26,013 - onnx-tf - INFO - Fail to get since_version of GatherND in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:54:26,014 - onnx-tf - INFO - Unknown op ImageScaler in domain `ai.onnx`.
2020-07-05 21:54:26,015 - onnx-tf - INFO - Fail to get since_version of Range in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 2

Instructions for updating:
Create a `tf.sparse.SparseTensor` and use `tf.sparse.to_dense` instead.


In [32]:
print(tf_rep.inputs) # Input nodes to the model
print('-----')
print(tf_rep.outputs) # Output nodes from the model
print('-----')
print(tf_rep.tensor_dict) # All nodes in the model

['i0']
-----
['o0']
-----
{'classifier.bias': <tf.Tensor 'classifier.bias:0' shape=(27,) dtype=float32>, 'classifier.weight': <tf.Tensor 'classifier.weight:0' shape=(27, 1280) dtype=float32>, 'features.0.0.weight': <tf.Tensor 'features.0.0.weight:0' shape=(32, 3, 3, 3) dtype=float32>, 'features.0.1.bias': <tf.Tensor 'features.0.1.bias:0' shape=(32,) dtype=float32>, 'features.0.1.num_batches_tracked': <tf.Tensor 'features.0.1.num_batches_tracked:0' shape=() dtype=int64>, 'features.0.1.running_mean': <tf.Tensor 'features.0.1.running_mean:0' shape=(32,) dtype=float32>, 'features.0.1.running_var': <tf.Tensor 'features.0.1.running_var:0' shape=(32,) dtype=float32>, 'features.0.1.weight': <tf.Tensor 'features.0.1.weight:0' shape=(32,) dtype=float32>, 'features.1.conv.0.weight': <tf.Tensor 'features.1.conv.0.weight:0' shape=(32, 1, 3, 3) dtype=float32>, 'features.1.conv.1.bias': <tf.Tensor 'features.1.conv.1.bias:0' shape=(32,) dtype=float32>, 'features.1.conv.1.num_batches_tracked': <tf.Tens

In [33]:
TF_MODEL_PATH = f'{MODELS_ROOT}/jestnet_stateful_tf.pb'
tf_rep.export_graph(TF_MODEL_PATH)

* Simplified -> TensorFlow:

In [34]:
tf_rep = prepare(model_simp)

2020-07-05 21:54:33,005 - onnx-tf - INFO - Fail to get since_version of BitShift in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:54:33,006 - onnx-tf - INFO - Unknown op ConstantFill in domain `ai.onnx`.
2020-07-05 21:54:33,006 - onnx-tf - INFO - Fail to get since_version of CumSum in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:54:33,007 - onnx-tf - INFO - Fail to get since_version of Det in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:54:33,008 - onnx-tf - INFO - Fail to get since_version of DynamicQuantizeLinear in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:54:33,009 - onnx-tf - INFO - Fail to get since_version of GatherND in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 21:54:33,009 - onnx-tf - INFO - Unknown op ImageScaler in domain `ai.onnx`.
2020-07-05 21:54:33,011 - onnx-tf - INFO - Fail to get since_version of Range in domain `` with max_inclusive_version=10. Set to 1.
2020-07-05 2

In [35]:
print(tf_rep.inputs) # Input nodes to the model
print('-----')
print(tf_rep.outputs) # Output nodes from the model
print('-----')
print(tf_rep.tensor_dict) # All nodes in the model

['i0']
-----
['o0']
-----
{'classifier.bias': <tf.Tensor 'classifier.bias:0' shape=(27,) dtype=float32>, 'classifier.weight': <tf.Tensor 'classifier.weight:0' shape=(27, 1280) dtype=float32>, '341': <tf.Tensor '341:0' shape=(1,) dtype=int64>, '343': <tf.Tensor '343:0' shape=(1, 3, 56, 56) dtype=float32>, '372': <tf.Tensor '372:0' shape=(1,) dtype=int64>, '374': <tf.Tensor '374:0' shape=(1, 4, 28, 28) dtype=float32>, '395': <tf.Tensor '395:0' shape=(1,) dtype=int64>, '397': <tf.Tensor '397:0' shape=(1, 4, 28, 28) dtype=float32>, '426': <tf.Tensor '426:0' shape=(1,) dtype=int64>, '428': <tf.Tensor '428:0' shape=(1, 8, 14, 14) dtype=float32>, '449': <tf.Tensor '449:0' shape=(1,) dtype=int64>, '451': <tf.Tensor '451:0' shape=(1, 8, 14, 14) dtype=float32>, '472': <tf.Tensor '472:0' shape=(1,) dtype=int64>, '474': <tf.Tensor '474:0' shape=(1, 8, 14, 14) dtype=float32>, '503': <tf.Tensor '503:0' shape=(1,) dtype=int64>, '505': <tf.Tensor '505:0' shape=(1, 12, 14, 14) dtype=float32>, '526': <t

In [36]:
TF_SIMPLE_MODEL_PATH = f'{MODELS_ROOT}/jestnet_stateful_simple_tf.pb'
tf_rep.export_graph(TF_SIMPLE_MODEL_PATH)

* TensorFlow -> tf.js:

```
tensorflowjs_converter --input_format=tf_frozen_model --output_node_names='o0' ./jestnet_stateful_tf.pb ./jestnet_stateful_web
```

```
tensorflowjs_converter --input_format=tf_frozen_model --output_node_names='o0' ./jestnet_stateful_simple_tf.pb ./jestnet_stateful_simple_web
```

* TensorFlow model speed check:

In [37]:
tf.compat.v1.disable_eager_execution()

[Very useful link](https://blog.metaflow.fr/tensorflow-how-to-freeze-a-model-and-serve-it-with-a-python-api-d4f3596b3adc)

In [38]:
def load_graph(frozen_graph_filename):
    # We load the protobuf file from the disk and parse it to retrieve the 
    # unserialized graph_def
    with tf.io.gfile.GFile(frozen_graph_filename, "rb") as f:
        graph_def = tf.compat.v1.GraphDef()
        graph_def.ParseFromString(f.read())

    # Then, we import the graph_def into a new Graph and returns it 
    with tf.Graph().as_default() as graph:
        # The name var will prefix every op/nodes in your graph
        # Since we load everything in a new graph, this is not needed
        tf.import_graph_def(graph_def, name="prefix")
    return graph

In [39]:
TF_SIMPLE_MODEL_PATH = f'{MODELS_ROOT}/jestnet_stateful_simple_tf.pb'
graph = load_graph(TF_SIMPLE_MODEL_PATH)

In [40]:
dir(graph)

['_ControlDependenciesController',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_add_control_dependencies',
 '_add_device_to_stack',
 '_add_function',
 '_add_new_tf_operations',
 '_add_op',
 '_apply_device_functions',
 '_as_graph_def',
 '_as_graph_element_locked',
 '_attr_scope',
 '_attr_scope_map',
 '_auto_cast_variable_read_dtype',
 '_bcast_grad_args_cache',
 '_building_function',
 '_c_graph',
 '_check_not_finalized',
 '_collections',
 '_colocate_with_for_gradient',
 '_colocation_stack',
 '_container',
 '_control_dependencies_for_inputs',
 '_control_dependencies_stack',
 '_control_flow_context',
 '_copy_functions_to_graph_def',
 '_create_op_from_tf_operation',
 '

In [41]:
# graph.get_operations()

In [42]:
# i0 = tf.compat.v1.placeholder(tf.float32, shape=[None, 3, 224, 224], name="i0")
# i1 = tf.compat.v1.placeholder(tf.float32, shape=[None, 3, 56, 56], name="i1")
# i2 = tf.compat.v1.placeholder(tf.float32, shape=[None, 4, 28, 28], name="i2")
# i3 = tf.compat.v1.placeholder(tf.float32, shape=[None, 4, 28, 28], name="i3")
# i4 = tf.compat.v1.placeholder(tf.float32, shape=[None, 8, 14, 14], name="i4")
# i5 = tf.compat.v1.placeholder(tf.float32, shape=[None, 8, 14, 14], name="i5")
# i6 = tf.compat.v1.placeholder(tf.float32, shape=[None, 8, 14, 14], name="i6")
# i7 = tf.compat.v1.placeholder(tf.float32, shape=[None, 12, 14, 14], name="i7")
# i8 = tf.compat.v1.placeholder(tf.float32, shape=[None, 12, 14, 14], name="i8")
# i9 = tf.compat.v1.placeholder(tf.float32, shape=[None, 20, 7, 7], name="i9")
# i10 = tf.compat.v1.placeholder(tf.float32, shape=[None, 20, 7, 7], name="i10")

tf_buffer = [
    np.zeros([1, 3, 224, 224])
]

# We can verify that we can access the list of operations in the graph
# for op in graph.get_operations():
#     if ':' in op.name:
#         print(op.name)
#     if '/i' in op.name:
#         print(op.name)
#     if '/o' in op.name:
#         print(op.name)
#     # prefix/Placeholder/inputs_placeholder
#     # ...
#     # prefix/Accuracy/predictions

# We access the input and output nodes
tf_inputs = []
tf_outputs = []
for i in range(len(tf_buffer)):
    tf_inputs.append(graph.get_tensor_by_name(f'prefix/i{i}:0'))
    tf_outputs.append(graph.get_tensor_by_name(f'prefix/o{i}:0'))

In [43]:
with tf.compat.v1.Session(graph=graph) as sess:
    # Note: we don't nee to initialize/restore anything
    # There is no Variables in this graph, only hardcoded constants 
    output = sess.run(tf_outputs, feed_dict={
        tf_inputs[i]: tf_buffer[i] for i in range(len(tf_buffer))
    })
    print(len(output))

1


In [44]:
with tf.compat.v1.Session(graph=graph) as sess:
    for _ in range(100):
        begin = time.time()
        output = sess.run(tf_outputs, feed_dict={
            tf_inputs[i]: tf_buffer[i] for i in range(len(tf_buffer))
        })
        print('Curr time (s):', time.time() - begin)

Curr time (s): 11.846487998962402
Curr time (s): 0.10794281959533691
Curr time (s): 0.10206389427185059
Curr time (s): 0.10094523429870605
Curr time (s): 0.11176776885986328
Curr time (s): 0.1158609390258789
Curr time (s): 0.10177421569824219
Curr time (s): 0.09836411476135254
Curr time (s): 0.09355902671813965
Curr time (s): 0.09694218635559082
Curr time (s): 0.0940542221069336
Curr time (s): 0.09174203872680664
Curr time (s): 0.09422898292541504
Curr time (s): 0.09452700614929199
Curr time (s): 0.0954289436340332
Curr time (s): 0.09228277206420898
Curr time (s): 0.09588980674743652
Curr time (s): 0.09113502502441406
Curr time (s): 0.09640192985534668
Curr time (s): 0.09451079368591309
Curr time (s): 0.09136271476745605
Curr time (s): 0.09744787216186523
Curr time (s): 0.10995197296142578
Curr time (s): 0.10039710998535156
Curr time (s): 0.09462881088256836
Curr time (s): 0.09226202964782715
Curr time (s): 0.10370516777038574
Curr time (s): 0.09814810752868652
Curr time (s): 0.0912022

* `torchvision.models.mobilenet_v2` sanity check:

In [45]:
mobilenetv2_torch = torchvision.models.mobilenet_v2(pretrained=False, num_classes=27)
mobilenetv2_torch

MobileNetV2(
  (features): Sequential(
    (0): ConvBNReLU(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): ConvBNReLU(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): ConvBNReLU(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=Tr

In [46]:
torch_module

MobileNetV2(
  (features): ModuleList(
    (0): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
        (3): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
       

In [36]:
MV2_MODEL_PATH = f'./models/mobilenet_v2.onnx'

torch_inputs = (torch.rand(1, 3, 224, 224, dtype=torch.float32))

for param in mobilenetv2_torch.parameters():
    param = param.float()

for module in mobilenetv2_torch.children():
    for module_1 in module.children():
        for module_2 in module_1.children():
            for module_3 in module_2.children():
                if hasattr(module_3, 'num_batches_tracked'):
                    module_3.num_batches_tracked = module_3.num_batches_tracked.float()
#                 if str(module_3).split('(')[0] == 'BatchNorm2d':
#                     print(module_3)

torch2onnx(
    torch_module=mobilenetv2_torch, 
    torch_inputs=torch_inputs, 
    onnx_path=MV2_MODEL_PATH
)

In [37]:
with open(MV2_MODEL_PATH, 'rb') as onnx_model_file:
    onnx_model = onnx.load_model(onnx_model_file)
onnx.checker.check_model(onnx_model)

In [38]:
# print(onnx.helper.printable_graph(onnx_model.graph))

In [39]:
model_simp, check = simplify(onnx_model)
assert check, "Simplified ONNX model could not be validated"

In [40]:
print('Before', onnx_model.ByteSize())
print('After', model_simp.ByteSize())

Before 9203331
After 8977491


In [41]:
# print(onnx.helper.printable_graph(model_simp.graph))

In [42]:
MV2_SIMPLE_MODEL_PATH = './models/mv2_simple.onnx'
onnx.save(model_simp, MV2_SIMPLE_MODEL_PATH)

In [43]:
ort_session = onnxruntime.InferenceSession(MV2_SIMPLE_MODEL_PATH)
ort_session

In [44]:
tf_rep = prepare(model_simp)
print(tf_rep.inputs) # Input nodes to the model
print('-----')
print(tf_rep.outputs) # Output nodes from the model
print('-----')
print(tf_rep.tensor_dict) # All nodes in the model

2020-06-28 23:38:30,849 - onnx-tf - INFO - Fail to get since_version of BitShift in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:38:30,850 - onnx-tf - INFO - Unknown op ConstantFill in domain `ai.onnx`.
2020-06-28 23:38:30,851 - onnx-tf - INFO - Fail to get since_version of CumSum in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:38:30,852 - onnx-tf - INFO - Fail to get since_version of Det in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:38:30,853 - onnx-tf - INFO - Fail to get since_version of DynamicQuantizeLinear in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:38:30,854 - onnx-tf - INFO - Fail to get since_version of GatherND in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:38:30,855 - onnx-tf - INFO - Unknown op ImageScaler in domain `ai.onnx`.
2020-06-28 23:38:30,857 - onnx-tf - INFO - Fail to get since_version of Range in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 2

['i0']
-----
['o0']
-----
{'classifier.1.bias': <tf.Tensor 'classifier.1.bias:0' shape=(27,) dtype=float32>, 'classifier.1.weight': <tf.Tensor 'classifier.1.weight:0' shape=(27, 1280) dtype=float32>, '467': <tf.Tensor '467:0' shape=(32, 3, 3, 3) dtype=float32>, '469': <tf.Tensor '469:0' shape=(32,) dtype=float32>, '471': <tf.Tensor '471:0' shape=(32, 1, 3, 3) dtype=float32>, '473': <tf.Tensor '473:0' shape=(32,) dtype=float32>, '475': <tf.Tensor '475:0' shape=(16, 32, 1, 1) dtype=float32>, '477': <tf.Tensor '477:0' shape=(16,) dtype=float32>, '479': <tf.Tensor '479:0' shape=(96, 16, 1, 1) dtype=float32>, '481': <tf.Tensor '481:0' shape=(96,) dtype=float32>, '483': <tf.Tensor '483:0' shape=(96, 1, 3, 3) dtype=float32>, '485': <tf.Tensor '485:0' shape=(96,) dtype=float32>, '487': <tf.Tensor '487:0' shape=(24, 96, 1, 1) dtype=float32>, '489': <tf.Tensor '489:0' shape=(24,) dtype=float32>, '491': <tf.Tensor '491:0' shape=(144, 24, 1, 1) dtype=float32>, '493': <tf.Tensor '493:0' shape=(144,

In [45]:
TF_SIMPLE_MODEL_PATH = './models/mv2_simple_tf.pb'
tf_rep.export_graph(TF_SIMPLE_MODEL_PATH)

* Keras -> tf.js MobileNetV2 sanity check:

In [47]:
keras_mv2 = tf.keras.applications.MobileNetV2(
    input_shape=None,
    alpha=1.4,
    include_top=True,
    weights=None,
    input_tensor=None,
    pooling=None,
    classes=27
)

Instructions for updating:
If using Keras pass *_constraint arguments to layers.


In [48]:
# keras_mv2.compile()
keras_mv2.summary()

Model: "mobilenetv2_1.40_224"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 3, 224, 224) 0                                            
__________________________________________________________________________________________________
Conv1_pad (ZeroPadding2D)       (None, 3, 225, 225)  0           input_1[0][0]                    
__________________________________________________________________________________________________
Conv1 (Conv2D)                  (None, 48, 112, 112) 1296        Conv1_pad[0][0]                  
__________________________________________________________________________________________________
bn_Conv1 (BatchNormalization)   (None, 48, 112, 112) 192         Conv1[0][0]                      
_______________________________________________________________________________

In [49]:
keras_mv2.save('./models/keras_mv2.h5')  # creates a HDF5 file 'my_model.h5'
del keras_mv2  # deletes the existing model

# returns a compiled model
# identical to the previous one
keras_mv2 = load_model('./models/keras_mv2.h5')
keras_mv2.summary()

Model: "mobilenetv2_1.40_224"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 3, 224, 224) 0                                            
__________________________________________________________________________________________________
Conv1_pad (ZeroPadding2D)       (None, 3, 225, 225)  0           input_1[0][0]                    
__________________________________________________________________________________________________
Conv1 (Conv2D)                  (None, 48, 112, 112) 1296        Conv1_pad[0][0]                  
__________________________________________________________________________________________________
bn_Conv1 (BatchNormalization)   (None, 48, 112, 112) 192         Conv1[0][0]                      
_______________________________________________________________________________

In [50]:
### NOTE: DIDN'T WORK!!! Use command line tool
# tfjs.converters.save_keras_model(keras_mv2, './models/keras_mv2/')

* MobileNetV2 with Shifts Keras implementation:

In [51]:
# kvar = tf.keras.backend.zeros((1,10), name='i0')
# ekvar = tf.keras.backend.eval(kvar)

[keras multiple inputs / outputs](https://github.com/tensorflow/tensorflow/issues/34114)

In [52]:
keras_buffer = [
    np.zeros([1, 3, 224, 224]),
    np.zeros([1, 3, 56, 56]),
    np.zeros([1, 4, 28, 28]),
    np.zeros([1, 4, 28, 28]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 12, 14, 14]),
    np.zeros([1, 12, 14, 14]),
    np.zeros([1, 20, 7, 7]),
    np.zeros([1, 20, 7, 7])
]

inputs = [
    layers.Input(keras_buffer[i].shape[1:], name=f'i{i}') 
    for i in range(len(keras_buffer))
]

outputs = [
    tf.keras.layers.Add()([
        tf.slice(inputs[i], [0,0,0,0], [-1,-1,5,5]), 
        tf.slice(inputs[i], [0,0,0,0], [-1,-1,5,5])
    ]) 
    for i in range(len(keras_buffer))
]

model = training.Model(inputs, outputs, name='model_1')

x = {f'i{i}': keras_buffer[i] for i in range(len(keras_buffer))}

y_pred = model(x)

In [53]:
print(len(y_pred))
print([y_pred[i].shape for i in range(len(y_pred))])

11
[TensorShape([1, 3, 5, 5]), TensorShape([1, 3, 5, 5]), TensorShape([1, 4, 5, 5]), TensorShape([1, 4, 5, 5]), TensorShape([1, 8, 5, 5]), TensorShape([1, 8, 5, 5]), TensorShape([1, 8, 5, 5]), TensorShape([1, 12, 5, 5]), TensorShape([1, 12, 5, 5]), TensorShape([1, 20, 5, 5]), TensorShape([1, 20, 5, 5])]


In [8]:
model.compile()
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
i0 (InputLayer)                 [(None, 3, 224, 224) 0                                            
__________________________________________________________________________________________________
i1 (InputLayer)                 [(None, 3, 56, 56)]  0                                            
__________________________________________________________________________________________________
i2 (InputLayer)                 [(None, 4, 28, 28)]  0                                            
__________________________________________________________________________________________________
i3 (InputLayer)                 [(None, 4, 28, 28)]  0                                            
____________________________________________________________________________________________

In [9]:
model.save('./models/add_model.h5')
del model
model = load_model('./models/add_model.h5')
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
i0 (InputLayer)                 [(None, 3, 224, 224) 0                                            
__________________________________________________________________________________________________
i1 (InputLayer)                 [(None, 3, 56, 56)]  0                                            
__________________________________________________________________________________________________
i2 (InputLayer)                 [(None, 4, 28, 28)]  0                                            
__________________________________________________________________________________________________
i3 (InputLayer)                 [(None, 4, 28, 28)]  0                                            
____________________________________________________________________________________________

In [12]:
tfjs.converters.save_keras_model(model, './models/add_model_2/')

/Users/izakharkin/opt/anaconda3/envs/tf20/lib/python3.7/site-packages/tensorflowjs/converters/keras_h5_conversion.py:122: H5pyDeprecationWarning: The default file mode will change to 'r' (read-only) in h5py 3.0. To suppress this warning, pass the mode you need to h5py.File(), or set the global default h5.get_config().default_file_mode, or set the environment variable H5PY_DEFAULT_READONLY=1. Available modes are: 'r', 'r+', 'w', 'w-'/'x', 'a'. See the docs for details.
  return h5py.File(h5file)


In [ ]:
{
    "class_name": "TensorFlowOpLayer", 
    "config": {
        "name": "Slice_10", 
        "trainable": true, 
        "dtype": "float32", 
        "node_def": {
            "name": "Slice_10", 
            "op": "Slice", 
            "input": ["i5", "Slice_10/begin", "Slice_10/size"], 
            "attr": {"Index": {"type": "DT_INT32"}, "T": {"type": "DT_FLOAT"}}}, 
        "constants": {"1": [0, 0, 0, 0], "2": [-1, -1, 5, 5]}}, 
    "name": "tf_op_layer_Slice_10", 
    "inbound_nodes": [[["i5", 0, 0, {}]]]
}


> `tensorflowjs_converter --input_format keras ./add_model.h5 ./add_model`

In [ ]:
{
    "class_name": "InputLayer", 
    "config": {
        "batch_input_shape": [null, 3, 224, 224], 
        "dtype": "float32", 
        "sparse": false, 
        "ragged": false, 
        "name": "i0"
    }, 
    "name": "i0", 
    "inbound_nodes": []
}

{
    "class_name": "TensorFlowOpLayer", 
    "config": {
        "name": "Slice", 
        "trainable": true, 
        "dtype": "float32", 
        "node_def": {
            "name": "Slice", 
            "op": "Slice", 
            "input": ["i0", "Slice/begin", "Slice/size"], 
            "attr": {"T": {"type": "DT_FLOAT"}, "Index": {"type": "DT_INT32"}}}, 
        "constants": {"1": [0, 0, 0, 0], "2": [-1, -1, 5, 5]}
    }, 
    "name": "tf_op_layer_Slice", 
    "inbound_nodes": [[["i0", 0, 0, {}]]]
},
{
    "class_name": "TensorFlowOpLayer", 
    "config": {
        "name": "Slice_1", 
        "trainable": true, 
        "dtype": "float32", 
        "node_def": {
            "name": "Slice_1", 
            "op": "Slice", 
            "input": ["i0", "Slice_1/begin", "Slice_1/size"], 
            "attr": {"T": {"type": "DT_FLOAT"}, "Index": {"type": "DT_INT32"}}}, 
        "constants": {"1": [0, 0, 0, 0], "2": [-1, -1, 5, 5]}}, 
    "name": "tf_op_layer_Slice_1", 
    "inbound_nodes": [[["i0", 0, 0, {}]]]
}, 
{
    "class_name": "TensorFlowOpLayer", 
    "config": {
        "name": "Slice_2", 
        "trainable": true, 
        "dtype": "float32", 
        "node_def": {
            "name": "Slice_2", 
            "op": "Slice", 
            "input": ["i1", "Slice_2/begin", "Slice_2/size"], 
            "attr": {"T": {"type": "DT_FLOAT"}, "Index": {"type": "DT_INT32"}}}, 
        "constants": {"1": [0, 0, 0, 0], "2": [-1, -1, 5, 5]}}, 
    "name": "tf_op_layer_Slice_2", 
    "inbound_nodes": [[["i1", 0, 0, {}]]]
}


* Correct slice model:

In [5]:
keras_buffer = [
    np.zeros([1, 3, 224, 224])
]

inputs = [
    layers.Input(keras_buffer[0].shape[1:], name=f'i{i}') 
    for i in range(len(keras_buffer))
]

# tf.keras.layers.Lambda(lambda x: x[:,:,:8,:8])(inputs[i])  # !!! Lambda didn't work woth tf.js converted
outputs = [
    tf.slice(inputs[i], [0,0,0,0], [-1,-1,8,8])
    for i in range(len(keras_buffer))
]

model = training.Model(inputs, outputs, name='model_1')


x = {f'i{i}': keras_buffer[i] for i in range(len(keras_buffer))}

y_pred = model(x)
print(len(y_pred))
print([y_pred[i].shape for i in range(len(y_pred))])

model.compile()
model.summary()

tfjs.converters.save_keras_model(model, './models/slice_model/')

AttributeError: module 'tensorflow.keras.backend' has no attribute 'slice'

* `from tensorflow.python import keras`
* `keras.utils.generic_utils.slice_arrays(input_a, start=0)`

* https://github.com/tensorflow/tensorflow/blob/e19e2d29d562724ead9e60e1ba4c4ffd91a0eb7a/tensorflow/python/keras/utils/generic_utils_test.py
    
* `from tensorflow.python.ops import array_ops`
* `array_ops.slice()  # same as tf.slice()`

* https://github.com/keras-team/keras/issues/890

* PyTorch -> Keras:

In [54]:
def pth2keras(pth_model, keras_model):
    m = {} # m : {'classifier.1.bias': np.array(...), ...}

    for k, v in pth_model.named_parameters():
        m[k] = v
    for k, v in pth_model.named_buffers(): # for batchnormalization
        m[k] = v
        
    print('torch model names:\n', m.keys())
    print('keras model names:')
    for layer in keras_model.layers:
        print(layer.name)

    with torch.no_grad():
        for layer in keras_model.layers:
            if isinstance(layer, DepthwiseConv2D):
                print(layer.name)
                weights = []
                weights.append(m[layer.name+'.weight'].permute(2, 3, 0, 1).data.numpy()) # weight
                if layer.use_bias:
                    weights.append(m[layer.name+'.bias'].data.numpy()) # bias
                layer.set_weights(weights)
            elif isinstance(layer, Conv2D):
                # https://github.com/keras-team/keras/issues/8144
                # pth: (out_ch, in_ch, h, w)
                # tf/keras: (h, w, in_ch, out_ch)
                print(layer.name)
                weights = []
#                 print(m[layer.name+'.weight'].shape)
                weights.append(m[layer.name+'.weight'].permute(2, 3, 1, 0).data.numpy()) # weight
                if layer.use_bias:
                    weights.append(m[layer.name+'.bias'].data.numpy()) # bias
                layer.set_weights(weights)
            elif isinstance(layer, BatchNormalization):
                print(layer.name)
                weights = []
                if layer.scale:
                    weights.append(m[layer.name+'.weight'].data.numpy()) # gamma
                if layer.center:
                    weights.append(m[layer.name+'.bias'].data.numpy()) # beta
                weights.append(m[layer.name+'.running_mean'].data.numpy()) # running_mean
                weights.append(m[layer.name+'.running_var'].data.numpy()) # running_var
                layer.set_weights(weights)
            elif isinstance(layer, Dense):
                print(layer.name)
                weights = []
                weights.append(m[layer.name+'.weight'].t().data.numpy())
                if layer.use_bias:
                    weights.append(m[layer.name+'.bias'].data.numpy())
                layer.set_weights(weights)

In [55]:
# from tensorflow.keras.layers import (
#     Conv2D, Dense, BatchNormalization, 
#     DepthwiseConv2D, Input, 
# )
# from tensorflow.keras.models import Model


# class PthModel(nn.Sequential):
#     def __init__(self):
#         super(PthModel, self).__init__(
#             nn.Conv2d(3, 1, kernel_size=3, stride=1, padding=1, bias=False)
#         )

# pth_model = PthModel()
# inputs = Input(shape=[224, 224, 3])
# outputs = Conv2D(
#     data_format='channels_last', 
#     filters=1, 
#     kernel_size=3, 
#     strides=1, 
#     padding='same', 
#     use_bias=False, 
#     name='0'
# )(inputs)
# keras_model = Model(inputs=inputs, outputs=outputs)

In [56]:
# keras_model.get_weights()[0].shape

In [57]:
# pth2keras(pth_model=pth_model, keras_model=keras_model)

In [58]:
# pth_inputs = torch.randn([1, 3, 224, 224], dtype=torch.float32)
# keras_inputs = pth_inputs.permute(0, 2, 3, 1).numpy()
# pth_outputs = pth_model(pth_inputs)
# pth_outputs = pth_outputs.permute(0, 2, 3, 1).data.numpy() # keras output 과 형태가 맞도록 변형
# keras_outputs = keras_model.predict(keras_inputs)
# print(np.abs(pth_outputs-keras_outputs).max())

MobileNetV2 check:

In [59]:
# mobilenetv2_torch = torchvision.models.mobilenet_v2(pretrained=False, width_mult=1.4, num_classes=27)
# torchsummary.summary(mobilenetv2_torch, input_size=(3, 224, 224))

In [60]:
# mobilenetv2_keras = tf.keras.applications.MobileNetV2(
#     input_shape=None,
#     input_tensor=None,
#     weights=None,
#     alpha=1.4,
#     include_top=True,
#     pooling=None,
#     classes=27
# )
# mobilenetv2_keras.summary()

In [61]:
# pth2keras(pth_model=mobilenetv2_torch, keras_model=mobilenetv2_keras)

In [64]:
### NOTE: DIDN'T WORK: 
# import pytorch2keras

In [65]:
# torch_inputs = (torch.rand(1, 3, 224, 224, dtype=torch.float32),
#                 torch.zeros([1, 3, 56, 56], dtype=torch.float32),
#                 torch.zeros([1, 4, 28, 28], dtype=torch.float32),
#                 torch.zeros([1, 4, 28, 28], dtype=torch.float32),
#                 torch.zeros([1, 8, 14, 14], dtype=torch.float32),
#                 torch.zeros([1, 8, 14, 14], dtype=torch.float32),
#                 torch.zeros([1, 8, 14, 14], dtype=torch.float32),
#                 torch.zeros([1, 12, 14, 14], dtype=torch.float32),
#                 torch.zeros([1, 12, 14, 14], dtype=torch.float32),
#                 torch.zeros([1, 20, 7, 7], dtype=torch.float32),
#                 torch.zeros([1, 20, 7, 7], dtype=torch.float32))

# k_model = pytorch2keras.pytorch_to_keras(
#     mobilenetv2_torch, 
#     args=torch_inputs[0],
#     verbose=True
# )

*NOTE: `/opt/anaconda3/envs/tf20/lib/python3.7/site-packages/onnx2keras/reshape_layers.py` was changed to fix the bugs!*

In [66]:
import onnx2keras
onnx_path = f'{MODELS_ROOT}/jestnet_stateful.onnx'
onnx_model = onnx.load(onnx_path)
k_model = onnx2keras.onnx_to_keras(
    onnx_model, 
    input_names=['i0'],
    name_policy='short'
)

INFO:onnx2keras:Converter is called.
DEBUG:onnx2keras:List input shapes:
DEBUG:onnx2keras:None
DEBUG:onnx2keras:List inputs:
DEBUG:onnx2keras:Input 0 -> i0.
DEBUG:onnx2keras:List outputs:
DEBUG:onnx2keras:Output 0 -> o0.
DEBUG:onnx2keras:Gathering weights to dictionary.
DEBUG:onnx2keras:Found weight classifier.bias with shape (27,).
DEBUG:onnx2keras:Found weight classifier.weight with shape (27, 1280).
DEBUG:onnx2keras:Found weight features.0.0.weight with shape (32, 3, 3, 3).
DEBUG:onnx2keras:Found weight features.0.1.bias with shape (32,).
DEBUG:onnx2keras:Found weight features.0.1.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.0.1.running_mean with shape (32,).
DEBUG:onnx2keras:Found weight features.0.1.running_var with shape (32,).
DEBUG:onnx2keras:Found weight features.0.1.weight with shape (32,).
DEBUG:onnx2keras:Found weight features.1.conv.0.weight with shape (32, 1, 3, 3).
DEBUG:onnx2keras:Found weight features.1.conv.1.bias with shape (32,).
DEBUG:o

DEBUG:onnx2keras:Found weight features.14.conv.4.running_mean with shape (576,).
DEBUG:onnx2keras:Found weight features.14.conv.4.running_var with shape (576,).
DEBUG:onnx2keras:Found weight features.14.conv.4.weight with shape (576,).
DEBUG:onnx2keras:Found weight features.14.conv.6.weight with shape (160, 576, 1, 1).
DEBUG:onnx2keras:Found weight features.14.conv.7.bias with shape (160,).
DEBUG:onnx2keras:Found weight features.14.conv.7.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.14.conv.7.running_mean with shape (160,).
DEBUG:onnx2keras:Found weight features.14.conv.7.running_var with shape (160,).
DEBUG:onnx2keras:Found weight features.14.conv.7.weight with shape (160,).
DEBUG:onnx2keras:Found weight features.15.conv.0.weight with shape (960, 160, 1, 1).
DEBUG:onnx2keras:Found weight features.15.conv.1.bias with shape (960,).
DEBUG:onnx2keras:Found weight features.15.conv.1.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.15.co

DEBUG:onnx2keras:Found weight features.3.conv.7.weight with shape (24,).
DEBUG:onnx2keras:Found weight features.4.conv.0.weight with shape (144, 24, 1, 1).
DEBUG:onnx2keras:Found weight features.4.conv.1.bias with shape (144,).
DEBUG:onnx2keras:Found weight features.4.conv.1.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.4.conv.1.running_mean with shape (144,).
DEBUG:onnx2keras:Found weight features.4.conv.1.running_var with shape (144,).
DEBUG:onnx2keras:Found weight features.4.conv.1.weight with shape (144,).
DEBUG:onnx2keras:Found weight features.4.conv.3.weight with shape (144, 1, 3, 3).
DEBUG:onnx2keras:Found weight features.4.conv.4.bias with shape (144,).
DEBUG:onnx2keras:Found weight features.4.conv.4.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.4.conv.4.running_mean with shape (144,).
DEBUG:onnx2keras:Found weight features.4.conv.4.running_var with shape (144,).
DEBUG:onnx2keras:Found weight features.4.conv.4.weight with 

DEBUG:onnx2keras:Found weight features.9.conv.7.num_batches_tracked with shape ().
DEBUG:onnx2keras:Found weight features.9.conv.7.running_mean with shape (64,).
DEBUG:onnx2keras:Found weight features.9.conv.7.running_var with shape (64,).
DEBUG:onnx2keras:Found weight features.9.conv.7.weight with shape (64,).
DEBUG:onnx2keras:Found input i0 with shape [3, 224, 224]
DEBUG:onnx2keras:######
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Converting ONNX operation
DEBUG:onnx2keras:type: Conv
DEBUG:onnx2keras:node_name: 315
DEBUG:onnx2keras:node_params: {'dilations': [1, 1], 'group': 1, 'kernel_shape': [3, 3], 'pads': [1, 1, 1, 1], 'strides': [2, 2], 'change_ordering': False, 'name_policy': 'short'}
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Check if all inputs are available:
DEBUG:onnx2keras:Check input 0 (name i0).
DEBUG:onnx2keras:Check input 1 (name features.0.0.weight).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:

DEBUG:onnx2keras:######
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Converting ONNX operation
DEBUG:onnx2keras:type: Conv
DEBUG:onnx2keras:node_name: 323
DEBUG:onnx2keras:node_params: {'dilations': [1, 1], 'group': 1, 'kernel_shape': [1, 1], 'pads': [0, 0, 0, 0], 'strides': [1, 1], 'change_ordering': False, 'name_policy': 'short'}
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Check if all inputs are available:
DEBUG:onnx2keras:Check input 0 (name 322).
DEBUG:onnx2keras:Check input 1 (name features.2.conv.0.weight).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:... found all, continue
DEBUG:onnx2keras:conv:Conv without bias
DEBUG:onnx2keras:conv:2D convolution
DEBUG:onnx2keras:Output TF Layer -> Tensor("323/Conv2D:0", shape=(None, 96, 112, 112), dtype=float32)
DEBUG:onnx2keras:######
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Converting ONNX operation
DEBUG:onnx2keras:type: BatchNormalization
DEBUG:onnx2keras:nod

DEBUG:onnx2keras:shape:Actual shape:
DEBUG:onnx2keras:shape:[None 24 56 56]
DEBUG:onnx2keras:Output TF Layer -> [None 24 56 56]
DEBUG:onnx2keras:######
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Converting ONNX operation
DEBUG:onnx2keras:type: Constant
DEBUG:onnx2keras:node_name: 332
DEBUG:onnx2keras:node_params: {'value': array(1), 'change_ordering': False, 'name_policy': 'short'}
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Check if all inputs are available:
DEBUG:onnx2keras:... found all, continue
DEBUG:onnx2keras:Output TF Layer -> 1
DEBUG:onnx2keras:######
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Converting ONNX operation
DEBUG:onnx2keras:type: Gather
DEBUG:onnx2keras:node_name: 333
DEBUG:onnx2keras:node_params: {'axis': 0, 'change_ordering': False, 'name_policy': 'short'}
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Check if all inputs are available:
DEBUG:onnx2keras:Check input 0 (name 331).
DEBUG:onnx2keras:Check input 1 (name 332).
DEBUG:onnx2keras:... found all, continue
DEBUG:onnx2keras:gather:Ga

OverflowError: Python int too large to convert to C long

In [11]:
import onnx2keras
onnx_path = f'{MODELS_ROOT}/jestnet_stateful_simple.onnx'
onnx_model = onnx.load(onnx_path)
k_model = onnx2keras.onnx_to_keras(
    onnx_model, 
    input_names=['i0'],
    name_policy='short'
)

INFO:onnx2keras:Converter is called.
DEBUG:onnx2keras:List input shapes:
DEBUG:onnx2keras:None
DEBUG:onnx2keras:List inputs:
DEBUG:onnx2keras:Input 0 -> i0.
DEBUG:onnx2keras:List outputs:
DEBUG:onnx2keras:Output 0 -> o0.
DEBUG:onnx2keras:Gathering weights to dictionary.
DEBUG:onnx2keras:Found weight classifier.bias with shape (27,).
DEBUG:onnx2keras:Found weight classifier.weight with shape (27, 1280).
DEBUG:onnx2keras:Found weight 341 with shape (1,).
DEBUG:onnx2keras:Found weight 343 with shape (1, 3, 56, 56).
DEBUG:onnx2keras:Found weight 372 with shape (1,).
DEBUG:onnx2keras:Found weight 374 with shape (1, 4, 28, 28).
DEBUG:onnx2keras:Found weight 395 with shape (1,).
DEBUG:onnx2keras:Found weight 397 with shape (1, 4, 28, 28).
DEBUG:onnx2keras:Found weight 426 with shape (1,).
DEBUG:onnx2keras:Found weight 428 with shape (1, 8, 14, 14).
DEBUG:onnx2keras:Found weight 449 with shape (1,).
DEBUG:onnx2keras:Found weight 451 with shape (1, 8, 14, 14).
DEBUG:onnx2keras:Found weight 472 

DEBUG:onnx2keras:Found weight 469 with shape (1,).
DEBUG:onnx2keras:Found weight 470 with shape (1,).
DEBUG:onnx2keras:Found weight 471 with shape (1,).
DEBUG:onnx2keras:Found weight 500 with shape (1,).
DEBUG:onnx2keras:Found weight 501 with shape (1,).
DEBUG:onnx2keras:Found weight 502 with shape (1,).
DEBUG:onnx2keras:Found weight 523 with shape (1,).
DEBUG:onnx2keras:Found weight 524 with shape (1,).
DEBUG:onnx2keras:Found weight 525 with shape (1,).
DEBUG:onnx2keras:Found weight 554 with shape (1,).
DEBUG:onnx2keras:Found weight 555 with shape (1,).
DEBUG:onnx2keras:Found weight 556 with shape (1,).
DEBUG:onnx2keras:Found weight 577 with shape (1,).
DEBUG:onnx2keras:Found weight 578 with shape (1,).
DEBUG:onnx2keras:Found weight 579 with shape (1,).
DEBUG:onnx2keras:Found input i0 with shape [3, 224, 224]
DEBUG:onnx2keras:######
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Converting ONNX operation
DEBUG:onnx2keras:type: Conv
DEBUG:onnx2keras:node_name: 315
DEBUG:onnx2keras:node_params: 

DEBUG:onnx2keras:Check input 0 (name 326).
DEBUG:onnx2keras:... found all, continue
DEBUG:onnx2keras:clip:Using ReLU(6.0) instead of clip
DEBUG:onnx2keras:Output TF Layer -> Tensor("328_1/Identity:0", shape=(None, 96, 56, 56), dtype=float32)
DEBUG:onnx2keras:######
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Converting ONNX operation
DEBUG:onnx2keras:type: Conv
DEBUG:onnx2keras:node_name: 329
DEBUG:onnx2keras:node_params: {'dilations': [1, 1], 'group': 1, 'kernel_shape': [1, 1], 'pads': [0, 0, 0, 0], 'strides': [1, 1], 'change_ordering': False, 'name_policy': 'short'}
DEBUG:onnx2keras:...
DEBUG:onnx2keras:Check if all inputs are available:
DEBUG:onnx2keras:Check input 0 (name 328).
DEBUG:onnx2keras:Check input 1 (name 688).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a numpy constant.
DEBUG:onnx2keras:Check input 2 (name 690).
DEBUG:onnx2keras:The input not found in layers / model inputs.
DEBUG:onnx2keras:Found in weights, add as a

OverflowError: Python int too large to convert to C long

In [57]:
ceil(5.5)

6

* **mmdnn pytorch2keras**: [does not support](https://github.com/microsoft/MMdnn/issues/794) multiple inputs, BUT YOU CAN CONCAT ALL INPUTS IN ONE TENSOR OF SHAPE  
`(1, sum(channels), 224, 224)` and then do slices to exctract the buffer inside of the model;
* mmdnn works [only with pytorch 0.4.0](https://github.com/microsoft/MMdnn/issues/426), I had a problem loading jester weights for pth0.4.0 so I stopped pushing this direction. 

**Looking towards pure keras tsm implementation.**

In [26]:
* https://kevin970401.github.io/etc/2019/08/21/converting-model-pth-keras.html
* https://github.com/tensorflow/tfjs/issues/2348
* https://www.npmjs.com/package/@tensorflow-models/handpose
* https://github.com/tensorflow/tfjs-models/tree/master/handpose

SyntaxError: invalid syntax (<ipython-input-26-9e7c6ff808a1>, line 1)

Can help: 
* [tf.js webcam demo](https://github.com/tensorflow/tfjs-examples/tree/master/webcam-transfer-learning)
* [tf.js native mobilenet](https://github.com/tensorflow/tfjs-models/tree/master/mobilenet)
* [tf.js-converter mobilenet demo](https://github.com/tensorflow/tfjs/tree/master/tfjs-converter/demo/mobilenet) (converted from TF)
* [onnx-tracing-vs-scripting](https://pytorch.org/docs/master/onnx.html#tracing-vs-scripting)
* [keras-js](https://github.com/transcranial/keras-js)

---

### Run model in camera loop:

* PyTorch loop:

In [1]:
# WINDOW_NAME = 'Video Gesture Recognition'

# print("Open camera...")
# cap = cv2.VideoCapture(0)

# print(cap)

# # set a lower resolution for speed up
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 320)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 240)

# # env variables
# full_screen = False
# cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
# cv2.resizeWindow(WINDOW_NAME, 640, 480)
# cv2.moveWindow(WINDOW_NAME, 0, 0)
# cv2.setWindowTitle(WINDOW_NAME, WINDOW_NAME)


# t = None
# index = 0

# idx = 0
# history = [2, 2]
# history_logit = []
# history_timing = []

# i_frame = -1

# print("Ready!")
# while True:
#     i_frame += 1
#     _, img = cap.read()  # (480, 640, 3) 0 ~ 255
#     if i_frame % 2 == 0:  # skip every other frame to obtain a suitable frame rate
#         t1 = time.time()
#         img_tran = torch.from_numpy(transform(img)).float().contiguous()
#         with torch.no_grad():
#             outputs = torch_module(img_tran)
#             feat = outputs
#         if SOFTMAX_THRES > 0:
#             feat_np = feat.numpy().reshape(-1)
#             feat_np -= feat_np.max()
#             softmax = np.exp(feat_np) / np.sum(np.exp(feat_np))
#             print(max(softmax))
#             if max(softmax) > SOFTMAX_THRES:
#                 idx_ = np.argmax(feat.numpy(), axis=1)[0]
#             else:
#                 idx_ = idx
#         else:
#             idx_ = np.argmax(feat.numpy(), axis=1)[0]
#         if HISTORY_LOGIT:
#             history_logit.append(feat.numpy())
#             history_logit = history_logit[-12:]
#             avg_logit = sum(history_logit)
#             idx_ = np.argmax(avg_logit, axis=1)[0]
#         idx, history = process_output(idx_, history)
#         t2 = time.time()
#         print(f"{index} {categories[idx]}")
#         current_time = t2 - t1
#     img = cv2.resize(img, (640, 480))
#     img = img[:, ::-1]
#     height, width, _ = img.shape
#     label = np.zeros([height // 10, width, 3]).astype('uint8') + 255
#     cv2.putText(label, 'Prediction: ' + categories[idx],
#                 (0, int(height / 16)),
#                 cv2.FONT_HERSHEY_SIMPLEX,
#                 0.7, (0, 0, 0), 2)
#     cv2.putText(label, '{:.1f} Vid/s'.format(1 / current_time),
#                 (width - 170, int(height / 16)),
#                 cv2.FONT_HERSHEY_SIMPLEX,
#                 0.7, (0, 0, 0), 2)
#     img = np.concatenate((img, label), axis=0)
#     cv2.imshow(WINDOW_NAME, img)
#     key = cv2.waitKey(1)
#     if key & 0xFF == ord('q') or key == 27:  # exit
#         break
#     elif key == ord('F') or key == ord('f'):  # full screen
#         print('Changing full screen option!')
#         full_screen = not full_screen
#         if full_screen:
#             print('Setting FS!!!')
#             cv2.setWindowProperty(WINDOW_NAME, cv2.WND_PROP_FULLSCREEN,
#                                   cv2.WINDOW_FULLSCREEN)
#         else:
#             cv2.setWindowProperty(WINDOW_NAME, cv2.WND_PROP_FULLSCREEN,
#                                   cv2.WINDOW_NORMAL)
#     if t is None:
#         t = time.time()
#     else:
#         nt = time.time()
#         index += 1
#         t = nt

In [2]:
# cap.release()
# cv2.destroyAllWindows()

* ONNX Runtime loop:

In [3]:
# DIDN'T WORK WITH STATEFUL TSM !!!

In [4]:
# import numpy as np
# import onnxruntime
# import time
# import cv2


# def transform(frame: np.ndarray):
#     # 480, 640, 3, 0 ~ 255
#     frame = cv2.resize(frame, (224, 224))  # (224, 224, 3) 0 ~ 255
#     frame = frame / 255.0  # (224, 224, 3) 0 ~ 1.0
#     frame = np.transpose(frame, axes=[2, 0, 1])  # (3, 224, 224) 0 ~ 1.0
#     frame = np.expand_dims(frame, axis=0)  # (1, 3, 480, 640) 0 ~ 1.0
#     return frame

# def process_output(idx_, history):
#     # idx_: the output of current frame
#     # history: a list containing the history of predictions
#     if not REFINE_OUTPUT:
#         return idx_, history
#     max_hist_len = 20  # max history buffer
#     # mask out illegal action !!!
#     if idx_ in [7, 8, 21, 22, 3]:
#         idx_ = history[-1]
#     # use only single no action class
#     if idx_ == 0:
#         idx_ = 2
#     # history smoothing
#     if idx_ != history[-1]:
#         if not (history[-1] == history[-2]): #  and history[-2] == history[-3]):
#             idx_ = history[-1]
#     history.append(idx_)
#     history = history[-max_hist_len:]
#     return history[-1], history


# MODELS_ROOT = './stateful_models/'
# ONNX_SIMPLE_MODEL_PATH = f'{MODELS_ROOT}/jestnet_stateful_simple.onnx'
# ort_session = onnxruntime.InferenceSession(ONNX_SIMPLE_MODEL_PATH)
# input_names = ['i0']
# output_names = ['o0']

# SOFTMAX_THRES = 0
# HISTORY_LOGIT = True
# REFINE_OUTPUT = True

# WINDOW_NAME = 'Video Gesture Recognition'

# print("Open camera...")
# cap = cv2.VideoCapture(0)

# print(cap)

# # set a lower resolution for speed up
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 320)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 240)

# # env variables
# full_screen = False
# cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
# cv2.resizeWindow(WINDOW_NAME, 640, 480)
# cv2.moveWindow(WINDOW_NAME, 0, 0)
# cv2.setWindowTitle(WINDOW_NAME, WINDOW_NAME)


# categories = [
#     "Doing other things",  # 0
#     "Drumming Fingers",  # 1
#     "No gesture",  # 2
#     "Pulling Hand In",  # 3
#     "Pulling Two Fingers In",  # 4
#     "Pushing Hand Away",  # 5
#     "Pushing Two Fingers Away",  # 6
#     "Rolling Hand Backward",  # 7
#     "Rolling Hand Forward",  # 8
#     "Shaking Hand",  # 9
#     "Sliding Two Fingers Down",  # 10
#     "Sliding Two Fingers Left",  # 11
#     "Sliding Two Fingers Right",  # 12
#     "Sliding Two Fingers Up",  # 13
#     "Stop Sign",  # 14
#     "Swiping Down",  # 15
#     "Swiping Left",  # 16
#     "Swiping Right",  # 17
#     "Swiping Up",  # 18
#     "Thumb Down",  # 19
#     "Thumb Up",  # 20
#     "Turning Hand Clockwise",  # 21
#     "Turning Hand Counterclockwise",  # 22
#     "Zooming In With Full Hand",  # 23
#     "Zooming In With Two Fingers",  # 24
#     "Zooming Out With Full Hand",  # 25
#     "Zooming Out With Two Fingers"  # 26
# ]

# t = None
# index = 0

# idx = 0
# history = [2, 2]
# history_logit = []
# history_timing = []

# i_frame = -1

# print("Ready!")
# while True:
#     i_frame += 1
#     _, img = cap.read()  # (480, 640, 3) 0 ~ 255
#     if i_frame % 2 == 0:  # skip every other frame to obtain a suitable frame rate
#         t1 = time.time()
#         img_tran = transform(img).astype(np.float32)
#         outputs = ort_session.run(output_names, {'i0': img_tran})
#         feat = outputs[0]
#         if SOFTMAX_THRES > 0:
#             feat_np = feat.reshape(-1)
#             feat_np -= feat_np.max()
#             softmax = np.exp(feat_np) / np.sum(np.exp(feat_np))
#             print(max(softmax))
#             if max(softmax) > SOFTMAX_THRES:
#                 idx_ = np.argmax(feat, axis=1)[0]
#             else:
#                 idx_ = idx
#         else:
#             idx_ = np.argmax(feat, axis=1)[0]
#         if HISTORY_LOGIT:
#             history_logit.append(feat)
#             history_logit = history_logit[-12:]
#             avg_logit = sum(history_logit)
#             idx_ = np.argmax(avg_logit, axis=1)[0]
#         idx, history = process_output(idx_, history)
#         t2 = time.time()
#         print(f"{index} {categories[idx]}")
#         current_time = t2 - t1
#     img = cv2.resize(img, (640, 480))
#     img = img[:, ::-1]
#     height, width, _ = img.shape
#     label = np.zeros([height // 10, width, 3]).astype('uint8') + 255
#     cv2.putText(label, 'Prediction: ' + categories[idx],
#                 (0, int(height / 16)),
#                 cv2.FONT_HERSHEY_SIMPLEX,
#                 0.7, (0, 0, 0), 2)
#     cv2.putText(label, '{:.1f} Vid/s'.format(1 / current_time),
#                 (width - 170, int(height / 16)),
#                 cv2.FONT_HERSHEY_SIMPLEX,
#                 0.7, (0, 0, 0), 2)
#     img = np.concatenate((img, label), axis=0)
#     cv2.imshow(WINDOW_NAME, img)
#     key = cv2.waitKey(1)
#     if key & 0xFF == ord('q') or key == 27:  # exit
#         break
#     elif key == ord('F') or key == ord('f'):  # full screen
#         print('Changing full screen option!')
#         full_screen = not full_screen
#         if full_screen:
#             print('Setting FS!!!')
#             cv2.setWindowProperty(WINDOW_NAME, cv2.WND_PROP_FULLSCREEN,
#                                   cv2.WINDOW_FULLSCREEN)
#         else:
#             cv2.setWindowProperty(WINDOW_NAME, cv2.WND_PROP_FULLSCREEN,
#                                   cv2.WINDOW_NORMAL)
#     if t is None:
#         t = time.time()
#     else:
#         nt = time.time()
#         index += 1
#         t = nt

In [5]:
# cap.release()
# cv2.destroyAllWindows()